# Импорты и пути

In [1]:
from pathlib import Path
import sys
import random
from collections import Counter
from itertools import product

import pandas as pd
from openai import AsyncOpenAI
from transformers import AutoTokenizer

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generation import GenerationResult, generate_one
from io_utils import (
    load_json,
    load_jsonl,
    load_checkpoint,
    save_checkpoint,
    load_results,
    save_json,
)
from prompts import build_messages
from validation import ValidationReason


DATA_PATH = PROJECT_ROOT / "data" / "dev.jsonl"

EXPERIMENT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "experiments"
    / "parameters"
)

CACHE_DIR = EXPERIMENT_DIR / "cache"
DIAGNOSTICS_DIR = EXPERIMENT_DIR / "diagnostics"
RESULTS_PATH = EXPERIMENT_DIR / "results.txt"
SAMPLE_PATH = EXPERIMENT_DIR / "experiment_sample.json"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

# Создание клиента

In [2]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

client = AsyncOpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)

In [3]:
response = await client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "user", "content": "Ответь кратко: какое слово считается самым смешным?"}
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

Существует множество смешных слов, но среди них нет одиночного "самого". Смысл смешного часто зависит от контекста и восприятия.


# Тестовая выборка из 20 товаров

- выборка из `dev.jsonl`;
- в тест входят только те товары, которые могут быть обработаны без ошибки long_data (вход <1500 токенов).

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def count_input_tokens(product: dict) -> int:
    """Считает токены полного входа модели."""
    messages = build_messages(source_product=product)

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True
    )

    return len(input_ids)


In [8]:
dev_data = load_jsonl(DATA_PATH)

if SAMPLE_PATH.exists():
    experiment_products = load_json(SAMPLE_PATH)

else:
    short_products = [
        product
        for product in dev_data
        if count_input_tokens(product) <= 1500
    ]

    rng = random.Random(42)

    experiment_products = rng.sample(
        short_products,
        20,
    )

    save_json(
        SAMPLE_PATH,
        experiment_products,
    )

print("Товаров в dev:", len(dev_data))
print("Товаров в эксперименте:", len(experiment_products))

Товаров в dev: 40
Товаров в эксперименте: 20


# Обработка всех товаров

In [4]:
async def run_products(
    products: list[dict],
    generation_params: dict,
    experiment_name: str,
) -> list[GenerationResult]:
    """Обрабатывает набор товаров с checkpoint для одной конфигурации."""

    cache_path = CACHE_DIR / f"{experiment_name}.jsonl"
    diagnostic_dir = DIAGNOSTICS_DIR / experiment_name

    processed_ids = load_checkpoint(cache_path)

    for product in products:
        if product["product_id"] in processed_ids:
            continue

        result = await generate_one(
            client=client,
            model_name=model_name,
            source_product=product,
            generation_params=generation_params,
            max_attempts=2,
            diagnostic_dir=diagnostic_dir,
        )

        save_checkpoint(
            cache_path,
            result.model_dump(mode="json"),
        )

    return [
        GenerationResult.model_validate(item)
        for item in load_results(cache_path)
    ]

# Расчет метрик

In [13]:
def summarize_experiment(
    experiment_name: str,
    generation_params: dict,
    results: list[GenerationResult],
) -> dict:
    """Считает метрики одной конфигурации генерации."""

    valid_results = [
        result
        for result in results
        if result.valid and result.card is not None
    ]

    n = len(results)

    input_tokens = sum(
        result.input_tokens
        for result in results
    )

    output_tokens = sum(
        result.output_tokens
        for result in results
    )

    avg_description_length = (
        sum(
            len(result.card.description)
            for result in valid_results
        )
        / len(valid_results)
        if valid_results
        else 0
    )

    output_truncated = sum(
        ValidationReason.OUTPUT_TRUNCATED in result.errors
        for result in results
    )

    return {
        "experiment": experiment_name,
        "temperature": generation_params["temperature"],
        "top_k": generation_params.get("top_k", "без ограничения"),
        "repetition_penalty": generation_params.get(
            "repetition_penalty",
            1.0,
        ),
        "max_tokens": generation_params["max_tokens"],
        "valid_rate": (
            len(valid_results) / n
            if n
            else 0
        ),
        "avg_attempts": (
            sum(result.attempts for result in results) / n
            if n
            else 0
        ),
        "avg_description_length": avg_description_length,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "avg_input_tokens": input_tokens / n if n else 0,
        "avg_output_tokens": output_tokens / n if n else 0,
        "output_truncated": output_truncated,
    }

# Запуск одной конфигурации параметров

In [14]:
async def run_experiment(
    experiment_name: str,
    generation_params: dict,
) -> dict:
    """Запускает одну конфигурацию и возвращает её метрики."""

    results = await run_products(
        products=experiment_products,
        generation_params=generation_params,
        experiment_name=experiment_name,
    )

    return summarize_experiment(
        experiment_name=experiment_name,
        generation_params=generation_params,
        results=results,
    )

# Сетка параметров

In [15]:
BASE_PARAMS = {
    "top_p": 0.9,
    "max_tokens": 450,
    "seed": 42,
}

temperature_values = [
    0.2,
    0.4,
    0.5,
]

top_k_values = [
    None,
    20,
    50,
]

repetition_penalty_values = [
    1.0,
    1.1,
    1.2,
]

parameter_results = []

for temperature, top_k, repetition_penalty in product(
    temperature_values,
    top_k_values,
    repetition_penalty_values,
):
    params = BASE_PARAMS.copy()

    params["temperature"] = temperature
    params["repetition_penalty"] = repetition_penalty

    if top_k is not None:
        params["top_k"] = top_k

    temperature_name = str(
        temperature
    ).replace(".", "")

    top_k_name = (
        "unlimited"
        if top_k is None
        else str(top_k)
    )

    repetition_name = str(
        repetition_penalty
    ).replace(".", "")

    experiment_name = (
        f"grid_temp_{temperature_name}"
        f"_top_k_{top_k_name}"
        f"_repetition_{repetition_name}"
    )

    summary = await run_experiment(
        experiment_name=experiment_name,
        generation_params=params,
    )

    parameter_results.append(summary)


parameter_df = pd.DataFrame(parameter_results)

display(
    parameter_df[
        [
            "temperature",
            "top_k",
            "repetition_penalty",
            "valid_rate",
            "avg_description_length",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_attempts",
            "output_truncated",
        ]
    ].sort_values(
        by=[
            "valid_rate",
            "avg_attempts",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

parameter_results_path = (
    EXPERIMENT_DIR
    / "temperature_top_k_repetition_results.txt"
)

parameter_df.to_csv(
    parameter_results_path,
    sep="\t",
    index=False,
)

,temperature,top_k,repetition_penalty,valid_rate,avg_description_length,avg_input_tokens,avg_output_tokens,avg_attempts,output_truncated
22,0.5,20,1.1,0.95,228.105263,1212.75,200.65,1.05,0
19,0.5,без ограничения,1.1,0.95,220.894737,1286.50,205.65,1.10,0
25,0.5,50,1.1,0.95,220.894737,1286.50,205.65,1.10,0
10,0.4,без ограничения,1.1,0.90,250.277778,1277.90,228.20,1.10,1
16,0.4,50,1.1,0.90,250.277778,1277.90,228.35,1.10,1
2,0.2,без ограничения,1.2,0.90,321.388889,1611.55,268.65,1.35,1
5,0.2,20,1.2,0.90,321.388889,1611.55,268.65,1.35,1
8,0.2,50,1.2,0.90,321.388889,1611.55,268.65,1.35,1
13,0.4,20,1.1,0.85,260.058824,1345.90,240.90,1.15,1
21,0.5,20,1.0,0.85,214.294118,1437.25,244.10,1.20,0


In [16]:
SELECTED_TEMPERATURE = 0.5
SELECTED_TOP_K = None
SELECTED_REPETITION_PENALTY = 1.1

In [17]:
final_params = {
    "temperature": SELECTED_TEMPERATURE,
    "top_p": 0.9,
    "repetition_penalty": SELECTED_REPETITION_PENALTY,
    "max_tokens": 450,
    "seed": 43,
}


final_experiment_name = (
    "final_temp_04"
    "_top_p_09"
    "_top_k_unlimited"
    "_repetition_11"
    "_seed_43"
)

final_summary = await run_experiment(
    experiment_name=final_experiment_name,
    generation_params=final_params,
)

final_result_path = (
    EXPERIMENT_DIR
    / "final_selected_parameters.txt"
)

final_text = f"""Финальный прогон параметров

Параметры:
temperature = {final_params["temperature"]}
top_p = {final_params["top_p"]}
top_k = без ограничения
repetition_penalty = {final_params["repetition_penalty"]}
max_tokens = {final_params["max_tokens"]}
seed = {final_params["seed"]}
max_attempts = 2
products = {len(experiment_products)}

Метрики:
valid_rate = {final_summary["valid_rate"]:.3f}
avg_attempts = {final_summary["avg_attempts"]:.3f}
avg_description_length = {final_summary["avg_description_length"]:.2f}
input_tokens = {final_summary["input_tokens"]}
output_tokens = {final_summary["output_tokens"]}
avg_input_tokens = {final_summary["avg_input_tokens"]:.2f}
avg_output_tokens = {final_summary["avg_output_tokens"]:.2f}
output_truncated = {final_summary["output_truncated"]}
"""

final_result_path.write_text(
    final_text,
    encoding="utf-8",
)

print(final_text)

Финальный прогон параметров

Параметры:
temperature = 0.5
top_p = 0.9
top_k = без ограничения
repetition_penalty = 1.1
max_tokens = 450
seed = 43
max_attempts = 2
products = 20

Метрики:
valid_rate = 0.900
avg_attempts = 1.150
avg_description_length = 227.33
input_tokens = 26749
output_tokens = 4496
avg_input_tokens = 1337.45
avg_output_tokens = 224.80
output_truncated = 1



In [ ]:
import prompts

current_system_prompt = prompts.SYSTEM_PROMPT

yesterday_system_prompt = current_system_prompt.replace(
    "4. Поле description формируй",
    "3. Поле description формируй",
).replace(
    "5. Если отзыв в reviews",
    "4. Если отзыв в reviews",
).replace(
    "6. Каждый пункт pros и cons",
    "5. Каждый пункт pros и cons",
).replace(
    "7. Не включай в pros или cons",
    "6. Не включай в pros или cons",
).replace(
    "8. В tags сочетай",
    "7. В tags сочетай",
)


prompts.SYSTEM_PROMPT = yesterday_system_prompt


control_params = {
    "temperature": 0.4,
    "top_p": 0.9,
    "max_tokens": 450,
    "seed": 42,
}

try:
    control_summary = await run_experiment(
        experiment_name="control_yesterday_settings",
        generation_params=control_params,
    )
finally:
    prompts.SYSTEM_PROMPT = current_system_prompt

pd.DataFrame([control_summary])

,experiment,top_k,repetition_penalty,max_tokens,valid_rate,avg_attempts,avg_description_length,input_tokens,output_tokens,avg_input_tokens,avg_output_tokens,output_truncated
0,control_yesterday_settings,без ограничения,1.0,450,0.75,1.3,229.066667,31602,5406,1580.1,270.3,0


# Выбор max_tokens

In [ ]:
max_tokens_values = [
    250,
    350,
    500,
]

max_tokens_results = []

for max_tokens in max_tokens_values:
    params = {
        "temperature": SELECTED_TEMPERATURE,
        "top_p": 0.9,
        "repetition_penalty": SELECTED_REPETITION_PENALTY,
        "max_tokens": max_tokens,
        "seed": 43,
    }

    if SELECTED_TOP_K is not None:
        params["top_k"] = SELECTED_TOP_K

    top_k_name = (
        "unlimited"
        if SELECTED_TOP_K is None
        else str(SELECTED_TOP_K)
    )

    repetition_name = str(
        SELECTED_REPETITION_PENALTY
    ).replace(".", "")

    experiment_name = (
        f"max_tokens_{max_tokens}"
        f"_top_k_{top_k_name}"
        f"_repetition_{repetition_name}"
    )

    summary = await run_experiment(
        experiment_name=experiment_name,
        generation_params=params,
    )

    max_tokens_results.append(summary)


max_tokens_df = pd.DataFrame(max_tokens_results)

display(
    max_tokens_df[
        [
            "max_tokens",
            "valid_rate",
            "avg_description_length",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_attempts",
            "output_truncated",
        ]
    ]
)

# Запись результатов

In [ ]:
all_results_df = pd.concat(
    [
        parameter_df,
        max_tokens_df,
    ],
    ignore_index=True,
)

all_results_df.to_csv(
    RESULTS_PATH,
    sep="\t",
    index=False,
)

all_results_df